In [ ]:
# pip install python-dotenv SQLAlchemy pymysql pandas unidecode

import os, re
import pandas as pd
from sqlalchemy import create_engine, text
from unidecode import unidecode
from dotenv import load_dotenv

# ========= 1) Kết nối MySQL qua biến môi trường (.env) =========
# .env ví dụ:
# DB_HOST=127.0.0.1
# DB_PORT=3306
# DB_USER=myuser
# DB_PASSWORD=mypass
# DB_NAME=mer_marketing_metrix_socials

load_dotenv()
host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
database = os.getenv("DB_NAME")
port = int(os.getenv("DB_PORT", "3306"))

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4")

# ========= 2) Truy vấn dữ liệu gốc =========
SQL = text(r"""
SELECT 
    DATE(pr.created_at) AS date_create,
    CASE
        WHEN pr.platform IN ('Shopee', 'Tiktok') THEN UPPER(pr.platform)
        WHEN pr.external_id IN ('KHO ECOM', 'ECOM SG') THEN 'FB/INS/ZL/NB'
        ELSE pr.external_id
    END AS store,
    CASE WHEN pr.cmt_type = 'store' THEN 'service' ELSE 'products' END AS type_rating,
    pr.rating,
    pr.comment
FROM product_ratings pr
WHERE 
    (DATE(pr.created_at) BETWEEN DATE_FORMAT(CURDATE(), '%Y-01-01') AND CURDATE() - INTERVAL 1 DAY)
    OR (DATE(pr.created_at) BETWEEN DATE_FORMAT(CURDATE() - INTERVAL 1 YEAR, '%Y-01-01')
                               AND DATE_SUB(CURDATE() - INTERVAL 1 DAY, INTERVAL 1 YEAR));
""")

df = pd.read_sql(SQL, engine)

# ========= 3) Chuẩn hóa và gắn motif bằng Python (gating) =========
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = unidecode(str(s).lower())
    s = re.sub(r"[^a-z0-9\s]", " ", s)        # bỏ ký tự đặc biệt
    s = re.sub(r"\s+", " ", s).strip()        # gộp khoảng trắng
    return s

def wb(token: str) -> str:
    # word-boundary nhẹ cho token ngắn
    return rf"(^|\s)({token})(\s|$)"

MOTIF_PATTERNS = {
    "__NEG__": [
        r"(khong|k)\s*hai\s*long", r"\bte\b", r"\btoi\b", r"\bxau\b",
        r"rach", r"bong", r"dau\s*chan", r"kho\s*chiu",
        r"(khong\s*tot|k\s*tot)", r"(khong\s*dep|k\s*dep)"
    ],
    # Logistics/Service
    "Giao hàng chậm":  [r"giao\s*hang\s*cham", r"ship\s*cham", r"giao\s*cham"],
    "Giao hàng nhanh": [r"giao\s*hang\s*nhanh", r"ship\s*nhanh", r"giao\s*nhanh"],
    "Đóng gói cẩn thận": [r"dong\s*goi", r"bao\s*goi", r"goi\s*hang\s*ky", r"can\s*than"],
    "Nhân viên nhiệt tình": [r"nhiet\s*tinh", r"tan\s*tam", r"than\s*thien", r"tu\s*van\s*tot"],
    # Product
    "Sản phẩm đẹp": [wb(r"dep"), r"san\s*pham\s*dep", r"rat\s*dep", r"dep\s*qua", r"mau\s*(dep|xinh)", r"nhin\s*sang"],
    "Êm/nhẹ, thoải mái": [wb(r"em"), wb(r"nhe"), r"(di|mang)\s*em", r"rat\s*em", r"rat\s*nhe"],
    "Form/size chuẩn": [r"vua\s*van", r"dung\s*size", r"dung\s*kich\s*co", r"form\s*chuan", r"chuan\s*size", r"om\s*chan"],
    "Form/size không vừa": [r"(form|size).*(chat|rong|kho\s*chiu|khong\s*vua\s*van)"],
    "Chất liệu tốt": [r"chat\s*lieu\s*(tot|to*t)", r"cao\s*cap", wb(r"ben"), wb(r"mem")],
    "Giá tốt": [r"gia\s*(tot|to*t)", wb(r"re"), r"hop\s*ly"],
    "Quà tặng/CSKH": [r"qua\s*tang", r"tang\s*kem", r"duoc\s*shop\s*tang"],
    # Positive chung
    "Hài lòng chung": [r"hai\s*long", r"ung\s*y", r"ok(e|ela|ie)?", wb(r"on"), r"tuyet\s*voi", r"rat\s*ok"],
}

PRIORITY = [
    "Không hài lòng",
    "Giao hàng chậm",
    "Form/size không vừa",
    "Giao hàng nhanh",
    "Đóng gói cẩn thận",
    "Nhân viên nhiệt tình",
    "Sản phẩm đẹp",
    "Êm/nhẹ, thoải mái",
    "Form/size chuẩn",
    "Chất liệu tốt",
    "Giá tốt",
    "Quà tặng/CSKH",
    "Hài lòng chung",
    "Khác/Không phân loại",
]

COMPILED = {k: [re.compile(p) for p in v] for k, v in MOTIF_PATTERNS.items()}

def extract_motifs(raw_comment: str, rating):
    text = normalize_text(raw_comment)
    found = set()

    is_neg = any(p.search(text) for p in COMPILED["__NEG__"])

    if is_neg:
        found.add("Không hài lòng")
        # ghi nhận “nguyên nhân tiêu cực” nếu có
        for motif in ["Giao hàng chậm", "Form/size không vừa"]:
            if any(p.search(text) for p in COMPILED[motif]):
                found.add(motif)
    else:
        # motifs dương + trung tính
        for motif, patterns in COMPILED.items():
            if motif == "__NEG__":
                continue
            if motif == "Hài lòng chung":
                # ví dụ: yêu cầu rating >= 4 mới ghi "Hài lòng chung"
                if any(p.search(text) for p in patterns) and (pd.isna(rating) or float(rating) >= 4):
                    found.add(motif)
            else:
                if any(p.search(text) for p in patterns):
                    found.add(motif)

    if not found:
        found = {"Khác/Không phân loại"}

    # Thứ tự ổn định để GROUP BY
    ordered = [m for m in PRIORITY if m in found]
    primary = ordered[0] if ordered else "Khác/Không phân loại"

    if "Không hài lòng" in found or "Giao hàng chậm" in found or "Form/size không vừa" in found:
        sentiment = "Negative"
    elif any(m in found for m in [
        "Giao hàng nhanh","Đóng gói cẩn thận","Nhân viên nhiệt tình","Sản phẩm đẹp",
        "Êm/nhẹ, thoải mái","Form/size chuẩn","Chất liệu tốt","Giá tốt","Quà tặng/CSKH","Hài lòng chung"
    ]):
        sentiment = "Positive"
    else:
        sentiment = "Neutral"

    return ",".join(ordered), primary, sentiment

df[["comment_motif","primary_motif","sentiment"]] = df.apply(
    lambda r: pd.Series(extract_motifs(r.get("comment"), r.get("rating"))), axis=1
)

# ========= 4) Dùng luôn cho báo cáo / lưu file =========
# ví dụ thống kê nhanh:
pivot = (df
         .groupby(["store","type_rating","primary_motif"], as_index=False)
         .size()
         .sort_values(["store","type_rating","size"], ascending=[True,True,False]))

print(df.head(3))
print(pivot.head(10))

# df.to_parquet("ratings_with_motif.parquet", index=False)
# df.to_csv("ratings_with_motif.csv", index=False, encoding="utf-8-sig")

In [7]:
df.to_excel('check_comment.xlsx')